# 02 - Feature Analysis

This notebook analyses the engineered features used for NRL match winner prediction.
We examine feature correlations, distributions, Elo rating trajectories, rolling form,
home advantage patterns, and preliminary feature importance.

**Sections:**
1. Load feature matrix
2. Feature correlation heatmap (top 30)
3. Feature distributions for key features
4. Elo rating trajectories over time
5. Rolling form features vs actual results
6. Home advantage analysis
7. Preliminary feature importance (mutual information)
8. Class balance and summary statistics

In [ ]:
# ============================================================
# Cell 1: Imports and load features parquet
# ============================================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from sklearn.feature_selection import mutual_info_classif

from config.settings import FEATURES_DIR, PROCESSED_DIR, START_YEAR, END_YEAR, RANDOM_SEED

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------------
# Load the feature matrix
# ------------------------------------------------------------------
features_path = FEATURES_DIR / "features.parquet"
if features_path.exists():
    features = pd.read_parquet(features_path)
    print(f"Loaded features: {features.shape[0]:,} rows x {features.shape[1]} columns")
else:
    # Fallback: try CSV or other names
    alt_paths = list(FEATURES_DIR.glob("*.parquet")) + list(FEATURES_DIR.glob("*.csv"))
    if alt_paths:
        fpath = alt_paths[0]
        features = pd.read_parquet(fpath) if fpath.suffix == ".parquet" else pd.read_csv(fpath)
        print(f"Loaded features from {fpath.name}: {features.shape[0]:,} rows x {features.shape[1]} cols")
    else:
        print(f"WARNING: No feature files found in {FEATURES_DIR}")
        print("Building features from processed data...")
        from processing.feature_engineering import build_all_features
        matches = pd.read_parquet(PROCESSED_DIR / "matches.parquet")
        ladders_path = PROCESSED_DIR / "ladders.parquet"
        ladders = pd.read_parquet(ladders_path) if ladders_path.exists() else None
        features = build_all_features(matches, ladders=ladders)
        print(f"Built features: {features.shape[0]:,} rows x {features.shape[1]} cols")

# Identify the target column
target_col = None
for candidate in ["home_win", "target", "result", "y"]:
    if candidate in features.columns:
        target_col = candidate
        break

if target_col is None and "home_score" in features.columns and "away_score" in features.columns:
    features["home_win"] = (features["home_score"] > features["away_score"]).astype(int)
    target_col = "home_win"

# Identify numeric feature columns (exclude metadata / identifiers)
META_COLS = {
    "home_team", "away_team", "venue", "date", "kickoff", "match_id",
    "season", "year", "round", "home_score", "away_score", target_col,
    "time_slot", "is_finals",
}
numeric_cols = [
    c for c in features.select_dtypes(include=[np.number]).columns
    if c not in META_COLS
]

print(f"Target column: {target_col}")
print(f"Numeric feature columns: {len(numeric_cols)}")
print(f"Season range: {features.get('season', features.get('year', pd.Series())).min()} - "
      f"{features.get('season', features.get('year', pd.Series())).max()}")

In [ ]:
# ============================================================
# Cell 2: Feature correlation heatmap (top 30 features)
# ============================================================

# Select top 30 features by absolute correlation with target
if target_col and len(numeric_cols) > 0:
    corr_with_target = (
        features[numeric_cols]
        .corrwith(features[target_col])
        .dropna()
        .abs()
        .sort_values(ascending=False)
    )

    top_n = min(30, len(corr_with_target))
    top_features = corr_with_target.head(top_n).index.tolist()

    print(f"Top {top_n} features by absolute correlation with '{target_col}':")
    for i, feat in enumerate(top_features[:15], 1):
        r = features[feat].corr(features[target_col])
        print(f"  {i:2d}. {feat:40s} r = {r:+.4f}")
    if top_n > 15:
        print(f"  ... and {top_n - 15} more")

    # Compute pairwise correlation matrix for top features
    corr_matrix = features[top_features].corr()

    fig, ax = plt.subplots(figsize=(16, 14))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        linewidths=0.5,
        annot_kws={"size": 6},
        ax=ax,
    )
    ax.set_title(f"Feature Correlation Heatmap (Top {top_n} by |r| with target)",
                 fontsize=14, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Cannot compute correlations: target or numeric features missing.")

In [ ]:
# ============================================================
# Cell 3: Feature distributions (histograms for key features)
# ============================================================

# Select a representative set of key features to visualize
key_feature_patterns = [
    "elo", "ladder_pos", "rolling", "days_rest", "win_pct",
    "form", "h2h", "odds", "career_apps", "venue_win_rate",
]

key_features = []
for pattern in key_feature_patterns:
    matches_found = [c for c in numeric_cols if pattern.lower() in c.lower()]
    if matches_found:
        key_features.append(matches_found[0])  # Take first match

# Also include top correlated features not yet in the list
if target_col:
    for feat in top_features[:12]:
        if feat not in key_features:
            key_features.append(feat)
        if len(key_features) >= 12:
            break

key_features = key_features[:12]  # Cap at 12 for a 3x4 grid

if key_features:
    n_cols = 4
    n_rows = (len(key_features) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
    axes = axes.ravel()

    for i, feat in enumerate(key_features):
        ax = axes[i]
        data = features[feat].dropna()

        if target_col:
            # Split by target class for comparison
            mask_valid = features[feat].notna() & features[target_col].notna()
            home_win = features.loc[mask_valid & (features[target_col] == 1), feat]
            away_win = features.loc[mask_valid & (features[target_col] == 0), feat]

            ax.hist(home_win, bins=30, alpha=0.6, label="Home Win", color="#1f77b4",
                    density=True, edgecolor="white")
            ax.hist(away_win, bins=30, alpha=0.6, label="Away Win", color="#ff7f0e",
                    density=True, edgecolor="white")
            ax.legend(fontsize=7)
        else:
            ax.hist(data, bins=30, color="#2ca02c", alpha=0.7, edgecolor="white", density=True)

        ax.set_title(feat, fontsize=9, fontweight="bold")
        ax.set_ylabel("Density", fontsize=8)
        ax.tick_params(labelsize=7)

    # Hide unused axes
    for j in range(len(key_features), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Feature Distributions (by match outcome)", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No key features found to visualize.")

In [ ]:
# ============================================================
# Cell 4: Elo rating trajectories over time
# ============================================================

# Look for Elo columns (home_elo, away_elo, or similar)
elo_col_home = None
elo_col_away = None
for c in features.columns:
    cl = c.lower()
    if "home" in cl and "elo" in cl and "diff" not in cl:
        elo_col_home = c
    if "away" in cl and "elo" in cl and "diff" not in cl:
        elo_col_away = c

season_col = "season" if "season" in features.columns else "year"

if elo_col_home and elo_col_away:
    print(f"Elo columns: {elo_col_home}, {elo_col_away}")

    # Build a team-level Elo timeseries
    elo_records = []
    date_col = "date" if "date" in features.columns else None

    for _, row in features.iterrows():
        date_val = row.get(date_col) if date_col else row.get(season_col)
        elo_records.append({"team": row.get("home_team"), "date": date_val, "elo": row.get(elo_col_home)})
        elo_records.append({"team": row.get("away_team"), "date": date_val, "elo": row.get(elo_col_away)})

    elo_df = pd.DataFrame(elo_records).dropna(subset=["team", "elo"])

    if date_col:
        elo_df["date"] = pd.to_datetime(elo_df["date"], errors="coerce")
        elo_df = elo_df.dropna(subset=["date"]).sort_values("date")

    # Get unique teams and assign colors
    teams = sorted(elo_df["team"].unique())
    palette = sns.color_palette("husl", len(teams))
    team_colors = dict(zip(teams, palette))

    fig, ax = plt.subplots(figsize=(18, 8))

    for team in teams:
        team_data = elo_df[elo_df["team"] == team]
        if date_col:
            ax.plot(team_data["date"], team_data["elo"],
                    label=team, color=team_colors[team], alpha=0.7, linewidth=1.2)
        else:
            ax.plot(team_data.index, team_data["elo"],
                    label=team, color=team_colors[team], alpha=0.7, linewidth=1.2)

    # Reference line at 1500 (starting Elo)
    ax.axhline(1500, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)

    ax.set_xlabel("Date" if date_col else "Match Index")
    ax.set_ylabel("Elo Rating")
    ax.set_title("Elo Rating Trajectories by Team", fontsize=14, fontweight="bold")
    ax.legend(
        bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, ncol=1,
        borderaxespad=0.0,
    )
    plt.tight_layout()
    plt.show()

    # Summary: top and bottom teams by latest Elo
    latest_elo = elo_df.groupby("team")["elo"].last().sort_values(ascending=False)
    print("\nCurrent Elo Rankings (most recent rating):")
    print("-" * 45)
    for i, (team, elo) in enumerate(latest_elo.items(), 1):
        print(f"  {i:2d}. {team:35s} {elo:.0f}")
else:
    print("Elo columns not found in features. Available columns with 'elo':")
    print([c for c in features.columns if "elo" in c.lower()])

In [ ]:
# ============================================================
# Cell 5: Rolling form features vs actual results
# ============================================================

# Find rolling form columns
rolling_cols = [c for c in numeric_cols if "rolling" in c.lower() or "form" in c.lower()]

if rolling_cols and target_col:
    print(f"Rolling/form columns found: {rolling_cols}")

    # Select up to 6 rolling features for visualization
    plot_cols = rolling_cols[:6]

    n_cols = 3
    n_rows = (len(plot_cols) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for i, col in enumerate(plot_cols):
        row_idx, col_idx = divmod(i, n_cols)
        ax = axes[row_idx, col_idx]

        valid = features[[col, target_col]].dropna()

        # Box plot split by target
        data_groups = [
            valid.loc[valid[target_col] == 0, col].values,
            valid.loc[valid[target_col] == 1, col].values,
        ]

        bp = ax.boxplot(
            data_groups,
            labels=["Away Win", "Home Win"],
            patch_artist=True,
            widths=0.6,
        )
        for patch, color in zip(bp["boxes"], ["#ff7f0e", "#1f77b4"]):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)

        ax.set_title(col, fontsize=10, fontweight="bold")
        ax.set_ylabel("Feature Value", fontsize=9)

        # Add mean markers
        for j, data in enumerate(data_groups, 1):
            if len(data) > 0:
                ax.scatter(j, np.mean(data), color="red", zorder=5, s=50, marker="D",
                          label="Mean" if j == 1 else "")

    # Hide unused axes
    for i in range(len(plot_cols), n_rows * n_cols):
        row_idx, col_idx = divmod(i, n_cols)
        axes[row_idx, col_idx].set_visible(False)

    fig.suptitle("Rolling Form Features by Match Outcome", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print(f"No rolling/form columns found. Available numeric columns: {numeric_cols[:20]}")

In [ ]:
# ============================================================
# Cell 6: Home advantage analysis - win rate by venue, by team
# ============================================================

if target_col and not features.empty:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # --- Panel 1: Home win rate by venue (top 15 venues by match count) ---
    ax = axes[0]
    if "venue" in features.columns:
        venue_wr = (
            features.dropna(subset=["venue", target_col])
            .groupby("venue")
            .agg(
                n_matches=(target_col, "count"),
                home_wr=(target_col, "mean"),
            )
            .query("n_matches >= 10")
            .sort_values("home_wr", ascending=True)
            .tail(15)
        )

        colors = ["#2ca02c" if wr > 0.5 else "#d62728" for wr in venue_wr["home_wr"]]
        ax.barh(venue_wr.index, venue_wr["home_wr"], color=colors, edgecolor="white")
        ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
        ax.set_xlabel("Home Win Rate")
        ax.set_title("Home Win Rate by Venue (min 10 games)", fontsize=11, fontweight="bold")
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

        for i, (venue, row) in enumerate(venue_wr.iterrows()):
            ax.text(row["home_wr"] + 0.01, i, f"{row['home_wr']:.0%} (n={row['n_matches']:.0f})",
                    va="center", fontsize=7)
    else:
        ax.text(0.5, 0.5, "No venue data", transform=ax.transAxes, ha="center")

    # --- Panel 2: Home win rate by team (when playing at home) ---
    ax = axes[1]
    if "home_team" in features.columns:
        team_home_wr = (
            features.dropna(subset=["home_team", target_col])
            .groupby("home_team")
            .agg(
                n_home=(target_col, "count"),
                home_wr=(target_col, "mean"),
            )
            .sort_values("home_wr", ascending=True)
        )

        colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(team_home_wr)))
        ax.barh(team_home_wr.index, team_home_wr["home_wr"], color=colors, edgecolor="white")
        ax.axvline(0.5, color="red", linestyle="--", linewidth=1)
        ax.set_xlabel("Home Win Rate")
        ax.set_title("Home Win Rate by Team", fontsize=11, fontweight="bold")
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

        for i, (team, row) in enumerate(team_home_wr.iterrows()):
            ax.text(row["home_wr"] + 0.01, i, f"{row['home_wr']:.0%}",
                    va="center", fontsize=7)

    plt.tight_layout()
    plt.show()

    # Overall home advantage stats
    overall = features[target_col].mean()
    print(f"\nOverall home win rate: {overall:.3f} ({overall:.1%})")
    print(f"Home advantage strength: {(overall - 0.5) * 100:+.1f} percentage points above 50%")
else:
    print("Target column or features not available for home advantage analysis.")

In [ ]:
# ============================================================
# Cell 7: Feature importance using mutual information
# ============================================================

if target_col and len(numeric_cols) > 0:
    # Prepare data: drop rows with any NaN in feature columns or target
    analysis_cols = [c for c in numeric_cols if features[c].notna().sum() > len(features) * 0.5]
    valid_mask = features[analysis_cols + [target_col]].notna().all(axis=1)
    X_mi = features.loc[valid_mask, analysis_cols].values
    y_mi = features.loc[valid_mask, target_col].values.astype(int)

    print(f"Computing mutual information for {len(analysis_cols)} features on {len(y_mi):,} samples...")

    # Compute mutual information scores
    mi_scores = mutual_info_classif(
        X_mi, y_mi,
        discrete_features=False,
        random_state=RANDOM_SEED,
        n_neighbors=5,
    )

    mi_df = (
        pd.DataFrame({"feature": analysis_cols, "mi_score": mi_scores})
        .sort_values("mi_score", ascending=False)
        .reset_index(drop=True)
    )

    # Display top 25
    top_mi = mi_df.head(25)

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.barh(
        range(len(top_mi)),
        top_mi["mi_score"].values,
        color=sns.color_palette("viridis", len(top_mi)),
        edgecolor="white",
    )
    ax.set_yticks(range(len(top_mi)))
    ax.set_yticklabels(top_mi["feature"].values, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Mutual Information Score")
    ax.set_title("Top 25 Features by Mutual Information with Target",
                 fontsize=14, fontweight="bold")

    # Annotate
    for i, v in enumerate(top_mi["mi_score"].values):
        ax.text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=8)

    plt.tight_layout()
    plt.show()

    # Print full ranking
    print("\nTop 15 Features by Mutual Information:")
    print("-" * 50)
    for i, row in mi_df.head(15).iterrows():
        print(f"  {i+1:2d}. {row['feature']:40s} MI = {row['mi_score']:.4f}")
else:
    print("Cannot compute mutual information: target or features missing.")

In [ ]:
# ============================================================
# Cell 8: Class balance check and summary statistics
# ============================================================

if target_col and not features.empty:
    print("CLASS BALANCE")
    print("=" * 50)
    class_dist = features[target_col].value_counts().sort_index()
    print(f"  Away wins (0): {class_dist.get(0, 0):>5,}  ({class_dist.get(0, 0) / len(features) * 100:.1f}%)")
    print(f"  Home wins (1): {class_dist.get(1, 0):>5,}  ({class_dist.get(1, 0) / len(features) * 100:.1f}%)")
    print(f"  Total matches: {len(features):>5,}")
    print(f"  Class ratio:   {class_dist.get(1, 0) / class_dist.get(0, 1):.3f}")

    # Pie chart of class distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel 1: Overall class balance
    ax = axes[0]
    labels = ["Away Win", "Home Win"]
    sizes = [class_dist.get(0, 0), class_dist.get(1, 0)]
    colors_pie = ["#ff7f0e", "#1f77b4"]
    ax.pie(sizes, labels=labels, colors=colors_pie, autopct="%1.1f%%",
           startangle=90, textprops={"fontsize": 11})
    ax.set_title("Overall Class Distribution", fontweight="bold")

    # Panel 2: Class balance by season
    ax = axes[1]
    season_balance = (
        features.dropna(subset=[target_col, season_col])
        .groupby(season_col)[target_col]
        .mean()
        .sort_index()
    )
    ax.plot(season_balance.index.astype(str), season_balance.values,
            marker="o", color="#1f77b4", linewidth=2, markersize=6)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=1, alpha=0.7)
    ax.fill_between(range(len(season_balance)), season_balance.values, 0.5,
                    alpha=0.15, color="blue")
    ax.set_xlabel("Season")
    ax.set_ylabel("Home Win Rate")
    ax.set_title("Home Win Rate Stability Across Seasons", fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    plt.xticks(rotation=45, ha="right")

    plt.tight_layout()
    plt.show()

    # Summary statistics for all numeric features
    print("\nFEATURE SUMMARY STATISTICS")
    print("=" * 50)
    summary = features[numeric_cols].describe().T
    summary["missing_%"] = (features[numeric_cols].isnull().sum() / len(features) * 100).round(1)
    summary["skew"] = features[numeric_cols].skew().round(3)
    display(
        summary[["count", "mean", "std", "min", "50%", "max", "missing_%", "skew"]]
        .style.format(
            {"count": "{:.0f}", "mean": "{:.3f}", "std": "{:.3f}",
             "min": "{:.3f}", "50%": "{:.3f}", "max": "{:.3f}",
             "missing_%": "{:.1f}", "skew": "{:.3f}"}
        )
    )
else:
    print("Features or target not available for summary.")